# 🚀 Master Notebook: Huấn Luyện AI 2.0 Multi-Agent RAG Trên Kaggle Dual T4
### Tích hợp công thức SOTA từ Unsloth (Qwen-Embedding-4B), MoAI, CoREB và Turkish Legal RAG
- **Phần cứng**: Kaggle Dual Tesla T4 (`2 x 16GB = 32GB VRAM`)
- **Tối ưu**: `CachedMultipleNegativesRankingLoss` (GradCache) + LoRA FP16
- **Tiêu chuẩn**: Đánh giá thực nghiệm độc lập bằng `Recall@10`, `nDCG@10`, `MRR@10` (Ablation Study A1 -> A4)

In [ ]:
def patch_environment():
    """Patches missing triton.ops, removes invalid BNB_CUDA_VERSION overrides, and handles CPU/GPU fallbacks for Kaggle and Colab runtimes."""
    import os
    import sys
    import types

    # 1. Clean BNB_CUDA_VERSION if it causes missing libcudart.so
    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 2. Patch Triton ops mock for Triton 3+ compatibility
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    m_ops = types.ModuleType("triton.ops")
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 3. Bulletproof fallback for bitsandbytes if native CUDA libs fail or not present
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.44.1"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')


In [ ]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN ĐÃ ĐƯỢC CỐ ĐỊNH PHIÊN BẢN (PINNED SOTA STACK)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes \
    accelerate>=1.1.0 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops huggingface_hub

print('✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!')


In [ ]:
# BƯỚC 3: HUẤN LUYỆN CROSS-ENCODER RERANKER (RECIPE CoREB: LoRA r=16, alpha=16, nDCG@10)
import os, sys, urllib.request

# 1. Đảm bảo file train_reranker_lora.py luôn là bản vá mới nhất từ GitHub
dest_path = "/kaggle/working/training_lab/training/reranker/train_reranker_lora.py" if os.path.exists("/kaggle/working/training_lab") else "training_lab/training/reranker/train_reranker_lora.py"
raw_url = "https://raw.githubusercontent.com/ducmanh-jr/Multi-Agent_RAG/main/training_lab/training/reranker/train_reranker_lora.py"
try:
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    urllib.request.urlretrieve(raw_url, dest_path)
    print("✅ Đã cập nhật train_reranker_lora.py mới nhất từ GitHub!")
except Exception as e:
    print(f"ℹ️ Dùng file local ({e})")

py_bin = sys.executable
print('\n' + '='*80 + '\n🎯 [STAGE 1] HUẤN LUYỆN RERANKER CROSS-ENCODER VỚI HARD NEGATIVES\n' + '='*80)
mod = "training_lab.training.reranker.train_reranker_lora" if os.path.exists("training_lab") else "training.reranker.train_reranker_lora"
dpath = "training_lab/data/triplets/massive_reranker_pairs.jsonl" if os.path.exists("training_lab") else "data/triplets/massive_reranker_pairs.jsonl"
!{py_bin} -m {mod} --epochs 3 --lr 3e-5 --data {dpath}


In [ ]:
# BƯỚC 4: HUẤN LUYỆN EMBEDDING (RECIPE MoAI: LoRA r=32, CachedMNRLoss GradCache)
import os, sys, time, urllib.request

# 1. Cập nhật accelerate>=1.1.0 và sentence-transformers để tương thích 100% với Trainer
!pip install -q 'accelerate>=1.1.0' --upgrade sentence-transformers

# 2. Tự động tải bản vá train_embedding_lora.py mới nhất từ GitHub
dest_path = "/kaggle/working/training_lab/training/embedding/train_embedding_lora.py" if os.path.exists("/kaggle/working/training_lab") else "training_lab/training/embedding/train_embedding_lora.py"
raw_url = f"https://raw.githubusercontent.com/ducmanh-jr/Multi-Agent_RAG/main/training_lab/training/embedding/train_embedding_lora.py?t={int(time.time())}"
try:
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    req = urllib.request.Request(raw_url, headers={'User-Agent': 'Mozilla/5.0', 'Cache-Control': 'no-cache'})
    with urllib.request.urlopen(req) as resp, open(dest_path, 'wb') as out_f:
        out_f.write(resp.read())
    print("✅ Đã cập nhật thành công train_embedding_lora.py bản vá mới nhất!")
except Exception as e:
    print(f"ℹ️ Dùng file local ({e})")

py_bin = sys.executable
print('\n' + '='*80 + '\n⚡ [STAGE 2] HUẤN LUYỆN EMBEDDING VỚI GRADCACHE (EFFECTIVE BATCH = 64)\n' + '='*80)
mod = "training_lab.training.embedding.train_embedding_lora" if os.path.exists("training_lab") else "training.embedding.train_embedding_lora"
dpath = "training_lab/data/triplets/massive_embedding_triplets.jsonl" if os.path.exists("training_lab") else "data/triplets/massive_embedding_triplets.jsonl"
!{py_bin} -m {mod} --epochs 3 --lr 3e-5 --data {dpath}


In [ ]:
# BƯỚC 5: HUẤN LUYỆN AGENT 2 (DOCUMENT READER) TRÊN TẬP DOCLAYNET THẬT (10 NHÃN)
import os, sys, time, urllib.request

# 1. Tự động đồng bộ bản vá Agent 2 mới nhất từ GitHub
dest_path = "/kaggle/working/training_lab/agent2_reader/src/trainer.py" if os.path.exists("/kaggle/working/training_lab") else "training_lab/agent2_reader/src/trainer.py"
raw_url = f"https://raw.githubusercontent.com/ducmanh-jr/Multi-Agent_RAG/main/training_lab/agent2_reader/src/trainer.py?t={int(time.time())}"
try:
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    req = urllib.request.Request(raw_url, headers={'User-Agent': 'Mozilla/5.0', 'Cache-Control': 'no-cache'})
    with urllib.request.urlopen(req) as resp, open(dest_path, 'wb') as out_f:
        out_f.write(resp.read())
    print("✅ Đã đồng bộ mã nguồn Agent 2 Reader mới nhất!")
except Exception as e:
    print(f"ℹ️ Dùng file local ({e})")

print('\n' + '='*80 + '\n📄 [STAGE 3] HUẤN LUYỆN LAYOUTLMv3 FOCAL LOSS TRÊN BẢNG & CÔNG THỨC THẬT\n' + '='*80)
target = "training_lab/agent2_reader/train_agent2_t4x2.py" if os.path.exists("training_lab") else "agent2_reader/train_agent2_t4x2.py"
!{sys.executable} {target} --epochs 3


In [ ]:
import sys
# BƯỚC 6: CHẠY ABLATION STUDY THỰC NGHIỆM ĐỘC LẬP (A1 -> A4)
import os
print('\n' + '='*80 + '\n🏆 [STAGE 4] CHẠY ABLATION STUDY ĐỐI CHỨNG THEO CHUẨN TURKISH LEGAL RAG\n' + '='*80)
mod = "training_lab.evaluation.run_ablation_benchmark" if os.path.exists("training_lab") else "evaluation.run_ablation_benchmark"
!{sys.executable} -m {mod}



In [ ]:
import sys
# ==============================================================================
# BƯỚC 7: ĐÁNH GIÁ CHUẨN QUỐC TẾ (RAGTRIAD, RGB BENCHMARK & TỰ SỬA SAI)
# Đánh giá 4 chỉ số cấp thế giới: Negative Rejection, Noise Robustness, Groundedness, Latency
# ==============================================================================
import os
bm_script = "training_lab/evaluation/world_class_benchmark.py" if os.path.exists("training_lab") else "evaluation/world_class_benchmark.py"
!{sys.executable} {bm_script}



In [ ]:
# BƯỚC 8: TỰ ĐỘNG ĐÓNG GÓI TRỌNG SỐ VỀ TAB KAGGLE OUTPUT (HOẶC LOCAL OUTPUTS)
import shutil
from pathlib import Path

base_out = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
BUNDLE_DIR = base_out / 'AI20_Trained_Adapters'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

# Gom trọng số các adapter đã huấn luyện
search_dirs = [Path('training_lab'), Path('.')]
found_adapters = set()
for sdir in search_dirs:
    if sdir.exists():
        for a in sdir.glob('**/adapter_model.safetensors'):
            # Bỏ qua các checkpoint trung gian, chỉ lấy checkpoint xuất bản cuối cùng
            if 'checkpoint' in str(a).lower():
                continue
            found_adapters.add(a.resolve())

if not found_adapters:
    print('⚠️ [CẢNH BÁO] Chưa tìm thấy adapter_model.safetensors nào để gom.')

for a in found_adapters:
    # Tránh va chạm thư mục khi nhiều agent cùng lưu trong weights/
    if a.parent.name == "weights":
        agent_name = a.parent.parent.name
    elif a.parent.parent.name == "weights":
        agent_name = f"{a.parent.parent.parent.name}_{a.parent.name}"
    else:
        agent_name = a.parent.name
    dest = BUNDLE_DIR / agent_name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(a, dest / 'adapter_model.safetensors')
    # Sao chép toàn bộ config và tokenizer metadata nếu có
    for cfg_f in a.parent.glob('*.json'):
        shutil.copy2(cfg_f, dest / cfg_f.name)
    print(f'📦 Đã gom hoàn chỉnh adapter & tokenizer: {agent_name} -> {dest}')

OUT_ZIP = base_out / 'AI20_MultiAgent_Weights_DualT4.zip'
shutil.make_archive(str(OUT_ZIP).replace('.zip', ''), 'zip', str(BUNDLE_DIR))
print('=' * 80)
print(f'🎉 [THÀNH CÔNG] File zip đã sẵn sàng tải về tại: {OUT_ZIP}')
print('👉 Bạn có thể tải file này từ tab Output bên phải màn hình Kaggle!')
print('=' * 80)


In [ ]:
# BƯỚC 9: TỰ ĐỘNG TẢI FILE ZIP LÊN GOOGLE DRIVE (HOẶC TẢI VỀ 1-CLICK)
import os, shutil
from pathlib import Path

OUT_ZIP = Path("/kaggle/working/AI20_MultiAgent_Weights_DualT4.zip") if Path("/kaggle/working/AI20_MultiAgent_Weights_DualT4.zip").exists() else Path("./outputs/AI20_MultiAgent_Weights_DualT4.zip")
TARGET_FOLDER_ID = "1upmXzTIlxpcCVGU8Vp9ZGV3xNa1yu-Q6"

print('=' * 80)
print('🚀 [STAGE 9] XUẤT FILE TRỌNG SỐ RA GOOGLE DRIVE / MÁY TÍNH')
size_mb = round(OUT_ZIP.stat().st_size / (1024*1024), 2) if OUT_ZIP.exists() else 0
print(f'📦 Tệp xuất bản: {OUT_ZIP} (Dung lượng: {size_mb} MB)')
print('=' * 80)

# LỰA CHỌN 1: TẠO LINK TẢI 1-CLICK TRỰC TIẾP VỀ MÁY TÍNH (NHANH NHẤT TRÊN KAGGLE)
try:
    from IPython.display import FileLink, display
    print('\n📥 [LỰA CHỌN 1] Bấm vào link màu xanh bên dưới để tải trực tiếp file zip về máy tính:')
    display(FileLink(str(OUT_ZIP)))
except Exception as e:
    print(f'ℹ️ FileLink: {e}')

# LỰA CHỌN 2: NẾU ĐANG CHẠY TRÊN GOOGLE COLAB
if os.path.exists('/content/drive/MyDrive'):
    dest_drive = Path(f'/content/drive/MyDrive/AI20_MultiAgent_Weights_DualT4.zip')
    shutil.copy2(OUT_ZIP, dest_drive)
    print(f'\n🎉 [GOOGLE COLAB] Đã sao chép thành công vào Drive: {dest_drive}')

# LỰA CHỌN 3: TẢI TRỰC TIẾP LÊN THƯ MỤC GOOGLE DRIVE QUA PYDRIVE2
print(f'\n☁️ [LỰA CHỌN 3] Tải trực tiếp vào thư mục Google Drive: https://drive.google.com/drive/folders/{TARGET_FOLDER_ID}')
try:
    import subprocess
    subprocess.run(["pip", "install", "-q", "pydrive2"], check=False)
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive

    gauth = GoogleAuth()
    gauth.CommandLineAuth() # Click vào link hiện ra để cấp quyền Google và dán mã xác thực
    drive = GoogleDrive(gauth)

    file_metadata = {
        'title': OUT_ZIP.name,
        'parents': [{'id': TARGET_FOLDER_ID}]
    }
    gfile = drive.CreateFile(file_metadata)
    gfile.SetContentFile(str(OUT_ZIP))
    print(f'⏳ Đang tải {OUT_ZIP.name} lên Google Drive...')
    gfile.Upload()
    print(f'🎉 [THÀNH CÔNG] Đã tải hoàn tất file zip vào thư mục Google Drive của bạn!')
    print(f'👉 Link kiểm tra: https://drive.google.com/drive/folders/{TARGET_FOLDER_ID}')
except Exception as e:
    print(f'\nℹ️ Ghi chú: Nếu chưa cấu hình OAuth Google, bạn hãy dùng [LỰA CHỌN 1] để tải file zip về máy trong 1 click, sau đó kéo thả vào link Drive trên nhé! ({e})')
